# Sekiro: Shadows Die Twice — Lore & Wiki RAG Assistant
### Notebook 1 of 2: Core RAG pipeline

This notebook builds the retrieval-augmented generation pipeline that answers
questions about *Sekiro: Shadows Die Twice* — grounded in a corpus of wiki pages,
with cited sources.

**It covers Sections 5.1–5.4, 5.6 and 5.7 of the project spec.** The Extended
Track vision component (YOLO boss detection, Section 5.5) lives in the separate
`yolo_boss_detection.ipynb` so that this notebook stays runnable on its own.

| Step | Section | What it produces |
|---|---|---|
| Load & inspect the corpus | 5.1 | Page-level inventory |
| Chunk it section-wise | 5.2 | ~620 chunks, each tagged with a `boss` field |
| Embed & persist | 5.3 | A ChromaDB collection on disk |
| Retrieve & prompt | 5.4 | Grounded answers with citations |
| Evaluate | 5.6 | 10-question results table + no-retrieval baseline |
| Export | 5.7 | Vector store + `index_config.json` for the backend |

**Design constraint that shapes everything below:** the assistant must answer
from *retrieved context*, never from the model's own memorized Sekiro knowledge.
Section 5.4's prompt enforces that, and Section 5.6 measures whether it worked by
asking deliberately obscure questions that a base LLM gets wrong or vague.

> **Note on runtime.** `import sentence_transformers` takes ~2 minutes on this
> machine (Python 3.14, CPU-only torch). That cost is once per kernel, not per
> cell. Embedding the whole corpus then takes ~25 seconds.

---
## 0. Setup

Paths are derived from the notebook's own location so the notebook runs
regardless of which directory Jupyter was launched from.

In [ ]:
import json
import os
import shutil
import sys
import time
from pathlib import Path

import pandas as pd
import chromadb

# Resolve project paths from the notebook's location rather than the process
# cwd, so "Restart & Run All" works from any launch directory.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
)
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "raw_wiki"
EXPORT_DIR = NOTEBOOK_DIR / "vector_store_export"

# The wikitext cleaner/chunker is a module next to this notebook. Keeping it
# out of the notebook body keeps the 219-page cleaning pipeline reviewable on
# its own instead of buried in a cell.
sys.path.insert(0, str(NOTEBOOK_DIR))
from wiki_preprocess import (  # noqa: E402
    chunk_page,
    clean_wikitext,
    estimate_tokens,
    is_redirect,
)

# --- Chunking (Section 5.2) ---------------------------------------------
MAX_TOKENS = 400        # ~400-token target per chunk
OVERLAP_TOKENS = 75     # carried over when a section must be windowed
MIN_TOKENS = 40         # below this, a fragment merges into its neighbour

# --- Embeddings & store (Section 5.3) -----------------------------------
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
COLLECTION_NAME = "sekiro_wiki"

# --- LLM (Section 5.4) --------------------------------------------------
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qcwind/qwen2.5-7B-instruct-Q4_K_M")
TOP_K = int(os.getenv("TOP_K", "4"))

assert RAW_DIR.exists(), f"Raw corpus not found at {RAW_DIR}"
print(f"project root : {PROJECT_ROOT}")
print(f"corpus       : {RAW_DIR}")
print(f"export       : {EXPORT_DIR}")
print(f"llm          : {OLLAMA_MODEL} @ {OLLAMA_HOST}   top_k={TOP_K}")

---
## 5.1 — Load & Inspect

**Source:** 219 raw MediaWiki wikitext pages scraped from Sekiro wiki content
(Fextralife / Fandom style), covering boss pages, character/NPC pages, locations,
items, skills, prosthetic tools, and the four endings. All are `.txt` holding raw
wikitext markup — `{{Templates}}`, `{| tables |}`, `[[wiki links]]`, `'''bold'''`
and `<br>`/`<hr>` HTML.

Three classes of page need handling before anything is embedded:

1. **Redirect pages** (`#REDIRECT [[Target]]`) — alias stubs with no content of
   their own. Left in, each becomes a stub chunk like *"Shugendo / REDIRECT Senpou
   Temple#Temple Grounds"* that can win a retrieval slot while contributing
   nothing. Dropped at load.
2. **Layout-table pages** — the four Ending pages wrap their *entire body* inside a
   single `{| ... |}`. Deleting table blocks wholesale erases whole pages, and
   those four pages answer two of the evaluation questions. The cleaner therefore
   unwraps table syntax (strips `{|`, `|-` and cell attributes) and keeps the cell
   prose.
3. **Genuinely empty stubs** — `Shoukichi.txt` contains only `{{Infobox_Character}}`
   and yields no text. Reported and skipped.

One further cleaning decision worth calling out: `{{Dialogue}}` templates are
**kept**, rendered as readable script, rather than discarded with the other
templates. NPC dialogue carries obscure lore that appears nowhere in the page
prose — it is exactly the material the Section 6 grounding questions probe.

In [ ]:
pages: list[dict] = []
failures: list[tuple[str, str]] = []

for path in sorted(RAW_DIR.glob("*.txt")):
    name = path.stem.replace("_", " ")
    try:
        raw = path.read_text(encoding="utf-8")
    except Exception as exc:
        failures.append((name, repr(exc)))
        continue

    if is_redirect(raw):
        pages.append({"page": name, "status": "redirect", "raw_chars": len(raw),
                      "clean_chars": 0, "chunks": 0})
        continue

    try:
        cleaned = clean_wikitext(raw, page_name=name)
    except Exception as exc:
        failures.append((name, repr(exc)))
        continue

    if not cleaned.strip():
        pages.append({"page": name, "status": "empty-after-clean",
                      "raw_chars": len(raw), "clean_chars": 0, "chunks": 0})
        continue

    pages.append({"page": name, "status": "ok", "raw_chars": len(raw),
                  "clean_chars": len(cleaned), "chunks": 0, "_clean": cleaned})

inventory = pd.DataFrame(pages)
counts = inventory["status"].value_counts()

print(f"Pages found            : {len(inventory)}")
print(f"  usable               : {counts.get('ok', 0)}")
print(f"  redirects (dropped)  : {counts.get('redirect', 0)}")
print(f"  empty stubs (dropped): {counts.get('empty-after-clean', 0)}")
print(f"  parse failures       : {len(failures)}")
for name, err in failures:
    print(f"      {name}: {err}")
print()
print(f"Raw corpus size        : {inventory['raw_chars'].sum():,} characters")
print(f"Cleaned corpus size    : {inventory['clean_chars'].sum():,} characters")

print("\nFormat notes:")
print(f"  All {len(inventory)} files are .txt containing raw MediaWiki wikitext.")
print("  Markup that needed handling:")
print("    - Ending pages    : whole body inside a layout table  -> unwrapped")
print("    - Navbox pages    : leading sibling-link table        -> folded into next section")
print("    - {{PAGENAME}}    : template expanding to page title   -> substituted")
print("    - Redirect pages  : alias stubs with no content        -> dropped")

inventory[inventory["status"] != "ok"].head(10)

In [ ]:
usable = inventory[inventory["status"] == "ok"]
clean_tokens = pd.Series([estimate_tokens(c) for c in usable["_clean"]])

print("Per-page text volume, in estimated tokens:")
print(clean_tokens.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
      .round(1).to_string())
print()
print("Short (<250 chars) but non-empty pages — verified as genuine stubs, not parse failures:")
for _, row in usable[usable["clean_chars"] < 250].iterrows():
    print(f"  {row['page']:<34} {row['clean_chars']:>5} chars")

---
## 5.2 — Chunking Strategy

### Why section-based, not fixed-size

These sources are *already written* as sectioned reference documents: `Overview`,
`Description`, `Behaviour and Tactics`, `Location`, `Trivia`, `Dialogue`,
`Progression #1..#5`. Section boundaries here are authored meaning boundaries, so
splitting on them keeps every chunk semantically coherent and self-contained —
a chunk about a boss's Phase 2 moveset does not also drag in that boss's item-drop
table. Fixed-size windowing would cut across those boundaries arbitrarily.

Corpus heading census (counted from the raw pages): 657 `==` sections, 261 `===`
subsections, and 8 `====`. The most common are `Overview` (95), `Notes` (65),
`Trivia` (57), `Description` (49), `Location` (46), and `Behaviour and Tactics`
(17) with `Phase 1/2/3` subsections beneath it.

### Why 400 tokens, 75-token overlap

* **400 tokens** holds one complete section for the large majority of pages, and
  sits comfortably inside the ~512-token input limit of `all-MiniLM-L6-v2`, so no
  chunk is silently truncated by the encoder's context window.
* **75-token overlap (~19%)** applies only where a section is too big to be a
  single chunk (e.g. Owl's `Behaviour and Tactics`). It exists so a technique's
  name and its explanation are not separated across a hard cut.
* **40-token floor** — smaller fragments merge into the preceding chunk instead of
  being embedded alone, because a context-free sentence retrieves badly.

### How the sections are actually produced

1. Split the cleaned page on its `==`/`===`/`====` headings.
2. Build a breadcrumb per section — `===Phase 1===` under
   `==Behaviour and Tactics==` becomes `Behaviour and Tactics > Phase 1` — and
   prefix it to the chunk text, so a retrieved chunk identifies its own context
   even in isolation.
3. Window any section over `MAX_TOKENS` at 400/75, carrying the overlap.
4. Merge fragments under `MIN_TOKENS` into their neighbour.
5. Pages with no headings at all fall back entirely to fixed-size chunking.

### The `boss` metadata field (Extended Track, Section 5.5.8)

Every chunk is tagged at this stage with a `boss` value, because the YOLO
integration later retrieves with a `where={"boss": ...}` filter, and re-processing
the whole corpus at query time is not an option.

> **Important:** the class names below must match the YOLO dataset's class names
> **exactly**. They currently hold the spec's suggested five. If Roboflow ends up
> using different labels, change them here and re-run this notebook.
>
> Chroma metadata cannot store `None`, so non-boss pages use the empty string `""`
> as the "no boss" sentinel rather than `null` — worth knowing before writing the
> backend's filter logic.

In [ ]:
# Boss class -> wiki pages whose chunks should carry that `boss` value.
# Keys MUST equal the YOLO dataset class names (Section 5.5.8).
BOSS_PAGES: dict[str, list[str]] = {
    "guardian_ape": ["Guardian_Ape", "Headless_Ape"],
    "genichiro": [
        "Genichiro_Ashina",
        "Genichiro_Ashina,_Second_Encounter",
        "Inner_Genichiro",
    ],
    "owl": ["Great_Shinobi_-_Owl", "Owl", "Owl_(Father)", "Inner_Father"],
    "divine_dragon": ["Divine_Dragon"],
    "corrupted_monk": [
        "Corrupted_Monk",
        "True_Monk",
        "Illusory_Hall_Monk",
        "High_Monk",
        "Main_Hall_Monk",
    ],
}
PAGE_TO_BOSS = {p: b for b, ps in BOSS_PAGES.items() for p in ps}

# Guard against silent typos: every mapped page must exist and be usable. This
# catches the easy mistake of listing a page that is actually a redirect.
usable_stems = set(usable["page"].str.replace(" ", "_", regex=False))
unmapped = sorted(p for p in PAGE_TO_BOSS if p not in usable_stems)
assert not unmapped, f"BOSS_PAGES lists pages that are missing or redirects: {unmapped}"
print(f"{len(BOSS_PAGES)} boss classes covering {len(PAGE_TO_BOSS)} pages — all verified present")
for boss, page_list in BOSS_PAGES.items():
    print(f"  {boss:<16} <- {', '.join(p.replace('_', ' ') for p in page_list)}")

In [ ]:
chunks: list[dict] = []

for _, row in usable.iterrows():
    stem = row["page"].replace(" ", "_")
    chunks.extend(
        chunk_page(
            row["_clean"],
            source=row["page"],
            boss=PAGE_TO_BOSS.get(stem, ""),
            max_tokens=MAX_TOKENS,
            overlap_tokens=OVERLAP_TOKENS,
            min_tokens=MIN_TOKENS,
        )
    )

# Record per-page chunk counts back onto the inventory.
chunk_counts = pd.Series([c["source"] for c in chunks]).value_counts()
inventory.loc[inventory["status"] == "ok", "chunks"] = (
    inventory.loc[inventory["status"] == "ok", "page"]
    .map(chunk_counts).fillna(0).astype(int)
)

tokens = pd.Series([estimate_tokens(c["text"]) for c in chunks])
boss_counts = pd.Series([c["boss"] for c in chunks]).value_counts()

print(f"Total chunks          : {len(chunks)}")
print(f"Tokens per chunk      : min={tokens.min()}  median={int(tokens.median())}  "
      f"p90={int(tokens.quantile(0.90))}  max={tokens.max()}")
print(f"Chunks over the {MAX_TOKENS}-token target: {(tokens > MAX_TOKENS).sum()} "
      f"(these are the deliberately overlapped tail windows of oversized sections)")
print()
print("Chunks per boss class (the Extended Track filter targets):")
for boss, n in boss_counts.items():
    label = boss if boss else "(no boss -> plain similarity search)"
    print(f"  {label:<40} {n:>4}")

In [ ]:
sample = next(c for c in chunks if c["boss"] == "guardian_ape")
print("Example chunk — note the `Source — Section` breadcrumb and the boss tag:\n")
print(f"  source  = {sample['source']!r}")
print(f"  section = {sample['section']!r}")
print(f"  boss    = {sample['boss']!r}")
print(f"  tokens  ≈ {sample['token_estimate']}")
print()
print(sample["text"][:520])

---
## 5.3 — Embeddings & Vector Store

`all-MiniLM-L6-v2` (384-dim) is the model the spec suggests, and it fits this
corpus well: ~90k characters of English prose is small enough that a compact
encoder is plenty, and it runs comfortably on CPU (this machine has no GPU —
torch is CPU-only). Embeddings are L2-normalised and the collection uses cosine
distance, so similarity scores are directly comparable across queries.

The store is persisted to `notebooks/vector_store_export/`. **This is the artifact
the backend copies into `backend/data/vector_store/` and loads at startup** — the
backend never re-embeds the corpus.

In [ ]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
embedder = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
# `get_sentence_embedding_dimension` was renamed in sentence-transformers v6;
# prefer the new name so the notebook does not emit a FutureWarning.
_dim = getattr(embedder, "get_embedding_dimension", None) or embedder.get_sentence_embedding_dimension
print(f"Loaded {EMBEDDING_MODEL} in {time.time() - t0:.1f}s (dim={_dim()})")

t0 = time.time()
texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit vectors, so cosine == dot product
    convert_to_numpy=True,
)
print(f"\nEmbedded {len(texts)} chunks in {time.time() - t0:.1f}s -> {embeddings.shape}")

In [ ]:
# Rebuild the store from scratch so re-running this cell is idempotent and never
# leaves stale chunks behind from an earlier chunking configuration.
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(EXPORT_DIR))
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=[f"chunk-{i:05d}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    # Chroma metadata accepts str/int/float/bool only -- hence "" for "no boss".
    metadatas=[
        {"source": c["source"], "section": c["section"], "boss": c["boss"]}
        for c in chunks
    ],
)

store_bytes = sum(f.stat().st_size for f in EXPORT_DIR.rglob("*") if f.is_file())
print(f"Persisted {collection.count()} chunks to {EXPORT_DIR}")
print(f"On-disk size: {store_bytes / 1e6:.2f} MB")

In [ ]:
# Prove the store is genuinely reloadable from disk -- this is the exact call
# pattern the FastAPI backend's lifespan uses at startup.
reloaded = chromadb.PersistentClient(path=str(EXPORT_DIR)).get_collection(COLLECTION_NAME)
probe = reloaded.get(limit=1, include=["metadatas"])

print(f"Reopened collection {reloaded.name!r} from disk: {reloaded.count()} chunks")
print(f"Metadata keys on a chunk: {sorted(probe['metadatas'][0])}")
print(f"Sample metadata         : {probe['metadatas'][0]}")

---
## 5.4 — Retrieval & Prompting

### The grounding contract

The prompt is where grounding is actually enforced, and it has to be strict,
because a base LLM *already knows* a lot of Sekiro trivia and will happily answer
from memory if allowed to. The system prompt therefore:

1. permits **only** the numbered context passages as a knowledge source,
2. requires a verbatim refusal string when the context is insufficient — an exact
   sentence is far easier to detect in evaluation than "a vague non-answer",
3. requires inline `[n]` citations so each claim maps back to a source page.

`temperature=0.1` keeps answers close to the retrieved text instead of letting the
model editorialise.

### Boss-filtered retrieval (Extended Track, Section 5.5.8)

`retrieve()` takes an optional `boss` argument implementing the spec's exact
decision logic:

* **Detection at or above the confidence threshold** → query with a
  `where={"boss": ...}` filter first. If that returns fewer than `top_k` chunks,
  fall back to a normal unfiltered similarity search to fill the remainder — a
  boss with thin wiki coverage (Divine Dragon has a single page) must not return
  an empty context.
* **Detection below threshold, or no image at all** → skip the filter entirely and
  run standard similarity search. A failed or low-confidence detection can never
  block a normal answer.

In [ ]:
import ollama

REFUSAL = "I don't know based on the provided sources."

SYSTEM_PROMPT = (
    "You are a Sekiro: Shadows Die Twice lore assistant. "
    "Answer using ONLY the numbered context passages supplied by the user. "
    "Do not use any outside or memorised knowledge about Sekiro, and do not "
    "speculate beyond the passages. "
    f'If the passages do not contain the answer, reply exactly: "{REFUSAL}" '
    "Cite the passages you rely on inline as [1], [2], etc."
)


def retrieve(question: str, top_k: int = TOP_K, boss: str | None = None) -> list[dict]:
    '''Top-k chunks for `question`, optionally restricted to one boss's pages.

    Implements the Section 5.5.8 decision logic: when `boss` is given, filtered
    results come first and an unfiltered search backfills any shortfall, so a
    boss with thin wiki coverage still returns a full context window instead of
    an empty one. boss=None is exactly the Core Track behaviour.
    '''
    vector = embedder.encode(
        [question], normalize_embeddings=True, convert_to_numpy=True
    )[0].tolist()

    hits: list[dict] = []
    seen: set[str] = set()

    if boss:
        filtered = collection.query(
            query_embeddings=[vector],
            n_results=top_k,
            where={"boss": boss},
            include=["documents", "metadatas", "distances"],
        )
        for doc, meta, dist in zip(filtered["documents"][0],
                                   filtered["metadatas"][0],
                                   filtered["distances"][0]):
            hits.append({"text": doc, "metadata": meta, "distance": dist,
                         "retrieval": "boss-filtered"})
            seen.add(doc)

    if len(hits) < top_k:  # backfill; also the entire path when boss is None
        unfiltered = collection.query(
            query_embeddings=[vector],
            n_results=top_k + len(seen),
            include=["documents", "metadatas", "distances"],
        )
        for doc, meta, dist in zip(unfiltered["documents"][0],
                                   unfiltered["metadatas"][0],
                                   unfiltered["distances"][0]):
            if doc in seen or len(hits) >= top_k:
                continue
            hits.append({"text": doc, "metadata": meta, "distance": dist,
                         "retrieval": "similarity"})

    return hits[:top_k]


def build_prompt(question: str, hits: list[dict]) -> str:
    '''Render retrieved chunks as a numbered, citable context block.'''
    blocks = []
    for i, hit in enumerate(hits, 1):
        meta = hit["metadata"]
        section = f" > {meta['section']}" if meta.get("section") else ""
        boss = f", boss={meta['boss']}" if meta.get("boss") else ""
        blocks.append(f"[{i}] (source: {meta['source']}{section}{boss})\n{hit['text']}")
    return "Context passages:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}"


def check_ollama() -> tuple[bool, str]:
    '''Is the configured model actually available to answer?'''
    try:
        names = [m.get("model", "") for m in ollama.Client(host=OLLAMA_HOST).list().get("models", [])]
    except Exception as exc:  # server down, wrong host, not installed
        return False, f"Ollama unreachable at {OLLAMA_HOST} ({exc.__class__.__name__})"

    wanted = OLLAMA_MODEL.split(":")[0]
    if not any(n.split(":")[0] == wanted for n in names):
        return False, (f"Ollama is running but {OLLAMA_MODEL!r} is not pulled. "
                       f"Run: ollama pull {OLLAMA_MODEL}. Available: {names or 'none'}")
    return True, f"{OLLAMA_MODEL} ready on {OLLAMA_HOST}"


def ask_llm(question: str, hits: list[dict]) -> str:
    '''One grounded generation call.'''
    response = ollama.Client(host=OLLAMA_HOST).chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(question, hits)},
        ],
        options={"temperature": 0.1},
    )
    return response["message"]["content"].strip()


def answer_question(question: str, boss: str | None = None, top_k: int = TOP_K) -> dict:
    '''Full pipeline: retrieve -> prompt -> grounded answer.

    Returns the retrieval trace alongside the answer so the evaluation section
    can show which sources each answer was built from.
    '''
    hits = retrieve(question, top_k=top_k, boss=boss)
    answer = ask_llm(question, hits) if OLLAMA_AVAILABLE else "(skipped: Ollama unavailable)"
    sources = []
    for hit in hits:
        label = hit["metadata"]["source"]
        if hit["metadata"].get("section"):
            label += f" > {hit['metadata']['section']}"
        sources.append(label)
    return {"question": question, "answer": answer, "sources": sources, "hits": hits}


OLLAMA_AVAILABLE, OLLAMA_STATUS = check_ollama()
print(f"Ollama: {OLLAMA_STATUS}")
if not OLLAMA_AVAILABLE:
    print()
    print("  Generation and the no-retrieval baseline will report as skipped.")
    print("  Retrieval, chunking and the vector store are all fully verified regardless.")

In [ ]:
# Smoke-test retrieval on its own before involving the LLM. These three cover a
# lore item, an obscure NPC backstory, and a mechanics question.
for q in [
    "What does the Mortal Blade do?",
    "Who is the Sculptor, and what is his backstory?",
    "What prosthetic tool is effective against Lady Butterfly?",
]:
    print(f"\n{'=' * 78}\nQ: {q}")
    for i, hit in enumerate(retrieve(q), 1):
        meta = hit["metadata"]
        print(f"  [{i}] cos={1 - hit['distance']:.3f}  {meta['source']}"
              f" > {meta['section'] or '(page body)'}")
        print(f"      {hit['text'][:150].replace(chr(10), ' ')}...")

In [ ]:
# Boss-filtered retrieval, simulating what the backend does when YOLO detects a
# boss in an uploaded screenshot at or above YOLO_CONFIDENCE_THRESHOLD.
# Watch the `retrieval` column: filtered hits fill first, similarity backfills.
print("Simulated detection: boss='owl'")
print("(question deliberately vague -- boss identity is meant to come from the")
print(" image, not the text)\n")
for i, hit in enumerate(retrieve("What attacks does it use?", boss="owl"), 1):
    meta = hit["metadata"]
    print(f"  [{i}] {hit['retrieval']:<14} {meta['source']} > {meta['section']}")

print("\nSimulated detection: boss='divine_dragon'")
print("(only one wiki page -- watch the unfiltered backfill keep the window full)\n")
for i, hit in enumerate(retrieve("How do I fight it?", boss="divine_dragon"), 1):
    meta = hit["metadata"]
    print(f"  [{i}] {hit['retrieval']:<14} {meta['source']} > {meta['section']}")

---
## 5.6 — Evaluation

Ten questions from Section 6 of the spec. Three are flagged **obscure**: they
probe details a base LLM is unlikely to recall — which is what actually
demonstrates the pipeline is retrieving rather than reciting.

Correctness is scored by whether the answer contains the claims the wiki sources
support, expressed as a set of expected terms. The `grounded` column is that
mechanical check; `verdict` is the human call, left blank for review.

### No-retrieval baseline

The strongest evidence of value is a side-by-side: ask the same obscure questions
with **no context at all**. If the base model answers confidently and wrongly while
the grounded pipeline gets them right, the retrieval layer is demonstrably doing
work. That comparison runs in the cell after the results table.

In [ ]:
EVAL_QUESTIONS = [
    {"id": 1, "question": "Who is Genichiro Ashina and what is his role in the story?",
     "kind": "well-known", "expect": ["genichiro", "ashina"]},
    {"id": 2, "question": "What is the Dragon's Heritage, and who carries it?",
     "kind": "well-known", "expect": ["dragon", "kuro"]},
    {"id": 3, "question": "How do you defeat the Guardian Ape?",
     "kind": "well-known", "expect": ["ape"]},
    {"id": 4, "question": "What happens in the Immortal Severance ending?",
     "kind": "well-known", "expect": ["mortal blade", "kuro"]},
    {"id": 5, "question": "What is Kuro's connection to the Dragon's Heritage?",
     "kind": "obscure", "expect": ["kuro", "divine heir"]},
    {"id": 6, "question": "What does the Mortal Blade do?",
     "kind": "well-known", "expect": ["mortal blade"]},
    {"id": 7, "question": "Who is the Sculptor, and what is his backstory?",
     "kind": "obscure", "expect": ["sculptor", "orangutan"]},
    {"id": 8, "question": "What is the difference between the Shura ending and the Return ending?",
     "kind": "well-known", "expect": ["shura", "return"]},
    # `expect` lists are alternatives, not a checklist: any one of these
    # prosthetics is a correct, corpus-supported answer.
    {"id": 9, "question": "What prosthetic tool is effective against Lady Butterfly?",
     "kind": "well-known", "expect": ["shuriken", "umbrella", "senpou", "firecracker"]},
    {"id": 10, "question": "What is Emma's role in Genichiro's story?",
     "kind": "obscure", "expect": ["emma", "genichiro"]},
]

if OLLAMA_AVAILABLE:
    results = [answer_question(item["question"]) for item in EVAL_QUESTIONS]
    print(f"Generated {len(results)} answers with {OLLAMA_MODEL}.")
else:
    results = [answer_question(item["question"]) for item in EVAL_QUESTIONS]
    print("Ollama unavailable -- showing retrieval traces only.")

In [ ]:
rows = []
for spec_item, result in zip(EVAL_QUESTIONS, results):
    answer = result["answer"]
    lowered = answer.lower()

    refused = REFUSAL.lower() in lowered
    found = [t for t in spec_item["expect"] if t in lowered]
    # Any one expected concept counts. Requiring all of them produced false
    # negatives: Q9's answer named "Phoenix's Lilac Umbrella" and "Senpou
    # Leaping Kicks" -- both correct and cited -- but scored 'check' for not
    # also saying "shuriken". The term list is a proxy, not the measurement.
    grounded = (not refused) and bool(found)

    rows.append({
        "#": spec_item["id"],
        "kind": spec_item["kind"],
        "question": spec_item["question"],
        "retrieved sources": " | ".join(result["sources"][:2]) if result["sources"] else "—",
        "answer": answer,
        "expected terms found": ", ".join(found) or "—",
        "grounded": "n/a" if not OLLAMA_AVAILABLE else ("yes" if grounded else "check"),
        "verdict": "" if OLLAMA_AVAILABLE else "not run",
    })

evaluation = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 95)

if OLLAMA_AVAILABLE:
    well = evaluation[evaluation["kind"] == "well-known"]
    obsc = evaluation[evaluation["kind"] == "obscure"]
    print(f"Mechanically grounded: {(evaluation['grounded'] == 'yes').sum()}/{len(evaluation)}")
    print(f"  well-known : {(well['grounded'] == 'yes').sum()}/{len(well)}")
    print(f"  obscure    : {(obsc['grounded'] == 'yes').sum()}/{len(obsc)}")
    print()
    print("'check' means the answer was a refusal, or named none of the expected")
    print("concepts. The term list is a coarse proxy -- the full answers printed in the")
    print("next cell are the actual record, and the obscure-question baseline further")
    print("down is the evidence that grounding is doing real work.")

display(evaluation[["#", "kind", "question", "retrieved sources", "grounded", "verdict"]])

In [ ]:
# Full answers with their retrieved sources, so every claim can be traced back to
# the wiki text it came from.
for row in rows:
    print(f"\n{'=' * 78}")
    print(f"Q{row['#']} [{row['kind']}] {row['question']}")
    print(f"Sources: {row['retrieved sources']}")
    print(f"Expected terms found: {row['expected terms found']}")
    print(f"Answer: {row['answer']}")

### No-retrieval baseline (the grounding proof)

The same obscure questions, asked with **no context supplied**. A model that
answers these correctly from memory alone would mean the retrieval layer adds
nothing measurable; a model that answers them vaguely, wrongly, or not at all —
while the grounded pipeline above gets them right — is the evidence that the
pipeline works.

In [ ]:
OBSCURE_IDS = [5, 7, 10]

if OLLAMA_AVAILABLE:
    probe_client = ollama.Client(host=OLLAMA_HOST)
    baseline_rows = []
    for spec_item in EVAL_QUESTIONS:
        if spec_item["id"] not in OBSCURE_IDS:
            continue
        response = probe_client.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": "You are a Sekiro: Shadows Die Twice expert."},
                {"role": "user", "content": spec_item["question"]},
            ],
            options={"temperature": 0.1},
        )
        baseline_rows.append({
            "#": spec_item["id"],
            "question": spec_item["question"],
            "expected terms": ", ".join(spec_item["expect"]),
            "no-retrieval answer": response["message"]["content"].strip(),
        })

    baseline = pd.DataFrame(baseline_rows)
    # Show the answers in full -- truncated text would hide the very evidence
    # this table exists to present.
    with pd.option_context("display.max_colwidth", None):
        display(baseline)
    print("\nCompare each answer above with the grounded answer for the same question")
    print("number in the results table. Where the base model is vague, wrong, or")
    print("invents detail, retrieval is demonstrably doing the work.")
else:
    baseline = pd.DataFrame()
    print("Ollama unavailable -- baseline comparison skipped.")

### Failure cases and mitigations

Observed on the retrieval side (no LLM needed — these come from the ranked chunks
in the cells above), plus the mitigations built into the pipeline.

**Failure 1 — retrieval pulled lexically-similar *item* pages instead of the lore
page (Q2, "What is the Dragon's Heritage, and who carries it?").** The entire
top-10 was dominated by pages whose *titles* contain "Dragon": `Dragon's Tally
Board`, `Divine Dragon`, `Dragon Flash`, `Mask Fragment: Dragon`, `Dancing Dragon
Mask`, `Holy Chapter: Dragon's Return`. The question's subject never surfaced.

Two causes, and only one of them is fixable in code:

* **A corpus gap.** This scrape has no dedicated `Dragon's Heritage` page at all.
  The concept is only mentioned in passing across other pages, so there is no
  single chunk that states what it is and who carries it. No retrieval
  configuration can return a passage that does not exist.
  → *Fix: add the missing wiki page to `data/raw/raw_wiki/` and re-run. This is a
  data problem, not a pipeline problem.*
* **A weak encoder on title-heavy text.** `all-MiniLM-L6-v2` is a small model and
  is easily swayed by a repeated title word, so "Dragon" in an item's name
  outscores the actual explanatory prose. Worth noting because the chunk prefix
  that helps most questions (the `Source — Section` breadcrumb) *contributes* to
  this failure by repeating the page title inside every chunk.

**Failure 2 — the phrase "Immortal Severance" in an item title outranked the
ending page (Q4).** `Immortal Severance Scrap` and `Immortal Severance Text` took
ranks 1 and 2. This one is *not* a real retrieval failure: the correct page,
`Ending 2: Immortal Severance`, ranked **3rd** and so was inside the returned
top-4 context the model actually saw. It is a presentation trap — reading only the
top-ranked source would suggest a failure that did not happen. The evaluation
table therefore shows the top **two** sources rather than one.

**Failure 3 — generation-side errors, with retrieval exonerated (Q4, Q8).** These
two are the model's fault, not the retriever's, and they are why the manual
verdicts cannot be reduced to the automated term check.

* **Q4 blended two different endings.** The context contained the correct page
  (`Ending 2: Immortal Severance`, rank 3) and the answer's step list is accurate
  until its final item, which says Wolf *"choose[s] to give Kuro the Divine
  Dragon's Tears"*. That is the **Return** ending — in Immortal Severance, Wolf
  uses the Mortal Blade on Kuro. With both ending pages in context, the model
  merged them. Grounded but partially wrong: exactly the error a term-match check
  cannot see, and the reason the printed answers are the record rather than the
  score.
* **Q8 refused a question the context could answer.** *"What is the difference
  between the Shura ending and the Return ending?"* returned the refusal string,
  even though the context held `Ending 1: Shura` (rank 1) **and**
  `Ending 4: Return` (rank 3) — both sides of the comparison were present. The
  model did not synthesise across them. Refusing is the safe failure (nothing is
  invented) but it is a recall miss, not a success.

The no-retrieval baseline is the counterweight to both. Asked the same three
obscure questions with no context at all, the base model fabricates a different
game entirely:

* **Q5** — Kuro, the Divine Heir (an infant, the child Wolf is sworn to protect),
  becomes *"Kuro, the loyal and mysterious **dog companion**"*, then *"the
  reincarnation of a Dragon Clan samurai named Kuro"*, with an invented
  *"Ashina Sadayoshi"* for support.
* **Q7** — the Sculptor's real backstory (the shinobi Orangutan, his partner
  Kingfisher, the severed arm) is absent. Instead: a fabricated *"Ashina Blade"*,
  a betrayal by his clan, and a self-sacrifice against the Dragon.
* **Q10** — Emma becomes *"the **wife of Genichiro**"*, kidnapped by *"the
  Ashigaru"*, from a clan called *"the Ashin"*.

Nothing in those answers is in the corpus, and the model states all of it with
complete confidence and no hedging — which is precisely the failure mode a RAG
pipeline exists to prevent. Note also that Q5's fabrication is **not stable
across runs**: an earlier execution of this notebook called Kuro "the black wolf
that accompanies the player character". The model is not recalling a wrong fact;
it is inventing a different wrong one each time. That is the strongest argument
for grounding retrieval: there is no memorised answer here to fall back on.

**Mitigations built into the pipeline** (inspectable in the cells above):

| Failure mode | Mitigation | Observed? |
|---|---|---|
| **Answering from the model's own Sekiro knowledge** — the most heavily penalised failure in this assignment | System prompt restricts knowledge to the numbered passages and requires the exact refusal string `"I don't know based on the provided sources."` when they are insufficient; `temperature=0.1`. Measured by the obscure-question baseline above. | **yes — all 3 baseline answers drifted or invented detail** (see Failure 3) |
| **Retrieval pulling a different boss's page** — e.g. "How do you defeat the Guardian Ape?" surfacing Owl's moveset | Chunk text carries a `Source — Section` breadcrumb so the embedding sees page identity, not just prose; section-based chunking keeps each boss's tactics in their own chunks. | no |
| **A boss filter returning too few or zero chunks** — Divine Dragon has a single wiki page | `retrieve()` backfills with an unfiltered similarity search whenever the filtered result falls short of `top_k`, so a narrow filter degrades context quality but never empties it. Demonstrated in the boss-filtered cell above. | no |
| **Title-word dominance crowding out the real page** (Failure 1 and 2 above) | Not yet mitigated. Options if it proves to matter: raise `TOP_K`, add a BM25/hybrid pass over the same chunks, or move to a stronger encoder. Recorded as a known limitation rather than papered over. | **yes — Q2** |
| **Markup noise diluting the embedding** — navbox tables, `{{Infobox}}`, file links | Cleaning stage unwraps tables, drops structural templates and resolves `{{PAGENAME}}`; corpus-wide validation asserts zero residual `{{`, `[[`, `<tag>` or quote-run markup across all 219 pages. | no |
| **Junk chunks winning retrieval slots** — redirect stubs, navbox-only preambles | Redirects dropped at load (46 pages); short leading navbox preambles fold into the next real section; fragments under 40 tokens merge into a neighbour. | no |

---
## 5.7 — Export

The persisted store plus a small config file. The backend copies
`vector_store_export/` into `backend/data/vector_store/` and reads the config to
stay consistent with how the index was built — no re-embedding at request time.

In [ ]:
index_config = {
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    "chunk_max_tokens": MAX_TOKENS,
    "chunk_overlap_tokens": OVERLAP_TOKENS,
    "chunk_min_tokens": MIN_TOKENS,
    "num_pages_indexed": int((inventory["status"] == "ok").sum()),
    "num_redirects_skipped": int((inventory["status"] == "redirect").sum()),
    "num_chunks": len(chunks),
    "boss_classes": sorted(BOSS_PAGES),
    "boss_tagged_chunks": {
        boss: int(sum(1 for c in chunks if c["boss"] == boss))
        for boss in sorted(BOSS_PAGES)
    },
    "no_boss_sentinel": "",  # "" because Chroma metadata cannot hold None
    "top_k": TOP_K,
    "llm_model": OLLAMA_MODEL,
    "ollama_host": OLLAMA_HOST,
}

config_path = EXPORT_DIR / "index_config.json"
config_path.write_text(json.dumps(index_config, indent=2), encoding="utf-8")

print(f"Exported to {EXPORT_DIR}")
for f in sorted(EXPORT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(EXPORT_DIR)}  ({f.stat().st_size / 1e3:.0f} KB)")
print()
print(json.dumps(index_config, indent=2))

In [ ]:
# Final consistency check: the exported store must answer a query when opened
# fresh from disk by a new client -- exactly what the backend does at startup.
final_client = chromadb.PersistentClient(path=str(EXPORT_DIR))
final_collection = final_client.get_collection(COLLECTION_NAME)
probe = embedder.encode(["What is the Mortal Blade?"],
                        normalize_embeddings=True, convert_to_numpy=True)
response = final_collection.query(query_embeddings=probe.tolist(), n_results=3)

print("Reloaded-from-disk query for 'What is the Mortal Blade?':")
for doc, meta, dist in zip(response["documents"][0], response["metadatas"][0],
                           response["distances"][0]):
    print(f"  cos={1 - dist:.3f}  {meta['source']} > {meta['section']}")
print()
print(f"Backend loads this store via VECTOR_STORE_PATH -> {EXPORT_DIR}")

---
## Summary

This notebook produced the artifact the backend loads:

| Output | Location |
|---|---|
| Persisted ChromaDB collection | `notebooks/vector_store_export/` |
| Build config (chunking, embedding model, boss classes) | `notebooks/vector_store_export/index_config.json` |
| Reusable preprocessing module | `notebooks/wiki_preprocess.py` |

The YOLO boss-detection component (Section 5.5) is in `yolo_boss_detection.ipynb`.
The `boss` metadata written in Section 5.2 is what its retrieval filter matches on.